# SSO Signup Optimization — Power Analysis

**Experiment:** SSO Signup Optimization (A/B/C) · **Owner:** Sergio Oyola  
**Primary metric:** Visit-to-Signup Rate · **Randomization unit:** `visitor_id` · **Allocation point:** signup page


## Allocation
Allocation is at the **signup page**, so the in-experiment population is **visitors who reach the signup page**, and the measurable per-arm conversion is **signups ÷ signup-page visitors** — per the PRD's "~45% drop-off", that baseline is **~55%**, **NOT** the 7% company funnel VSR. The 7% is kept only as funnel context as it cannot be measured inside the experiment because non-reachers are never allocated.

Signup-page reach is identified from **`curated.product_tracking_events`** (event-level `url`). This is event-level, unlike the session `landing_page` field, so it captures everyone who *reaches* the page, not just direct landers.


## A/B/C Design Choices
- 3 arms (Control / Treatment A / Treatment B), equal **1/3 split**. Per-arm n is unchanged by the split; **duration** grows because each arm gets only 1/3 of traffic.
- Two planned comparisons (A vs C, B vs C) | **Bonferroni** for *sizing*: `alpha=0.025` (0.05 / 2).
- Winner between A and B is picked on **point estimate** (higher VSR).
- **Two-sided** test: sizing for a +MDE gives symmetric power to detect harm of the same magnitude, so the Early Stop / harm threshold mirrors the commited MDE (same magnitude, opposite sign) — pending Product team sign-off on the MDE (expected ~3%).


## Multiple Comparison - Bonferroni vs. other approaches
**Bonferroni** controls **Type I** (family-wise false-positive) error; its cost is reduced power. **Holm-Bonferroni** controls the same FWER and is uniformly more powerful. **Dunnett's test** is even more powerful for this design (many treatments vs. one shared control, exploiting the known correlation between comparisons that arises from the shared control arm).

So why does this notebook size with plain Bonferroni? Two reasons:

- **Holm-Bonferroni** steps through ordered p-values at analysis time. Pre-experiment, the rank of any given comparison is unknown, so the threshold it will face is unknown. The only defensible planning assumption is the worst-case threshold — which is **α/m** -> Bonferroni.

- **Dunnett's test** has no standard closed-form sample size formula. Pre-experiment power under Dunnett requires simulation or specialized tables and is not a general-purpose planning tool. Bonferroni provides a closed-form, conservative bound that is always valid.

Both methods yield more power than Bonferroni *at analysis time*, but neither enables a legitimately smaller *n* at the planning stage.

In [ ]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

# Pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters
SIGNUP_URL_PATTERN = 'https://www.stitchfix.com/signup%'
START_DATE         = '2026-01-01'

# Design parameters
INITIAL_ALPHA = 0.05
N_COMPARISONS = 2 # A vs C, B vs C
ALPHA = INITIAL_ALPHA / N_COMPARISONS # = 0.025 Bonferroni
POWER = 0.80
TWO_SIDED = True
N_ARMS = 3
MDE_GRID = [0.03, 0.05, 0.08, 0.10] # relative lift on signup-page conversion
HARM_GRID = [-0.03, -0.05] # relative drop (Early Stop Condition)

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/traitlets/traitlets.py", line 651, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/traitlets/traitlets.py", line 706, in __get__
    return self.get(obj, cls)  # type:ignore[return-value]
  File "/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/traitlets/traitlets.py", line

## Step 1 — Signup-page traffic & baseline conversion (from PTE)

Per visitor: did they reach the signup page, and (among new visitors) did they sign up. Aggregated monthly to get **daily signup-page reachers** and the **signup-page conversion rate** (baseline).

In [ ]:
# ----------------------------------------------------------------------------------
# Signup-page conversion baseline at the signup-page-visit grain.
# One row per visitor's first signup-page visit each month; new-visitor and signup
# ... status are evaluated relative to that visit, so a month's numbers depend only
# ... on that month's visits.
# ----------------------------------------------------------------------------------

CONV_WINDOW_DAYS = 7 # signup counts if it occurs within N days of the visit (Reporter should match this 7-day window)

traffic_query = f"""--sql
WITH signup_page_visits AS (
    -- One row per visitor per month, anchored to their first signup-page visit that month.
    SELECT
        visitor_id,
        DATE_TRUNC('month', datetime_in_utc) AS month,
        MIN(datetime_in_utc) AS signup_page_ts -- visit anchor for the conversion checks below
    FROM curated.product_tracking_events
    WHERE date_in_utc >= DATE '{START_DATE}'
      AND url LIKE '{SIGNUP_URL_PATTERN}'
    GROUP BY visitor_id, DATE_TRUNC('month', datetime_in_utc)
),
visitor_signup AS (
    -- One signup timestamp per visitor (NULL if never signed up).
    -- This assumes signup_ts is populated on the visitor's in-window session rows.
    SELECT
        visitor_id,
        MAX(signup_ts) AS signup_ts
    FROM curated.user_session_conversion_metrics
    WHERE region = 'US'
      AND date_in_utc >= DATE '{START_DATE}'
    GROUP BY visitor_id
),
classified AS (
    -- Flag each visit: 1) was the visitor new at visit time; 2) did they sign up within the window.
    SELECT
        v.month,
        v.signup_page_ts,
        CASE WHEN s.signup_ts IS NULL OR s.signup_ts >= v.signup_page_ts
             THEN 1 ELSE 0 END AS new_visitor, -- not yet signed up when they reached the page
        CASE WHEN s.signup_ts >= v.signup_page_ts
              AND s.signup_ts <  v.signup_page_ts + INTERVAL '{CONV_WINDOW_DAYS}' DAY
             THEN 1 ELSE 0 END AS signup -- signed up within 7 days (CONV_WINDOW_DAYS) of the visit
    FROM signup_page_visits v
    LEFT JOIN visitor_signup s ON s.visitor_id = v.visitor_id
),
month_days AS (
    -- Distinct calendar days with a signup-page visit, used to average daily volume.
    SELECT month, COUNT(DISTINCT CAST(signup_page_ts AS DATE)) AS days_observed
    FROM signup_page_visits
    GROUP BY month
)
SELECT
    c.month,
    md.days_observed,
    SUM(new_visitor) AS signup_page_visitors,
    SUM(signup) AS signups,
    CAST(SUM(signup) AS DOUBLE) / NULLIF(SUM(new_visitor), 0) AS signup_page_conv_rate,
    ROUND(SUM(new_visitor) / md.days_observed, 1) AS signup_page_visitors_per_day
FROM classified c
JOIN month_days md ON c.month = md.month
GROUP BY c.month, md.days_observed
ORDER BY c.month DESC
"""

traffic_df = query(traffic_query)
traffic_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,signup_page_visitors,signups,signup_page_conv_rate,signup_page_visitors_per_day
0,2026-06-01 00:00:00.000,25,183988,95258,0.517740,7359
1,2026-05-01 00:00:00.000,31,269372,142107,0.527549,8689
2,2026-04-01 00:00:00.000,30,288936,153146,0.530034,9631
3,2026-03-01 00:00:00.000,31,368861,199718,0.541445,11898
4,2026-02-01 00:00:00.000,28,335801,184522,0.549498,11992
5,2026-01-01 00:00:00.000,31,358933,199154,0.554850,11578


In [ ]:
# Reference month = last complete month (May 2026).
REFERENCE_MONTH = '2026-05-01'
last = traffic_df[traffic_df['month'].astype(str).str.startswith(REFERENCE_MONTH)].reset_index(drop=True)

BASELINE_RATE = float(last['signup_page_conv_rate'][0])
DAILY_ELIGIBLE_VISITORS = float(last['signup_page_visitors_per_day'][0])

# PRD mentions 45% drop-off, meaning ~55% completion, BASELINE_RATE = 0.55
# Here we are using the actual observed baseline rate from the last complete month (May 2026)

print(f"BASELINE_RATE = {BASELINE_RATE:.4f}  |  DAILY_ELIGIBLE_VISITORS = {DAILY_ELIGIBLE_VISITORS:,.0f}")
last.T

BASELINE_RATE = 0.5275  |  DAILY_ELIGIBLE_VISITORS = 8,689


,0
month,2026-05-01 00:00:00.000
days_observed,31
signup_page_visitors,269372
signups,142107
signup_page_conv_rate,0.527549
signup_page_visitors_per_day,8689


## Step 2 — Sample size & duration

`n_total_statsmodels` is pairwise; we read `n_treatment` as the **per-arm** requirement (split 0.5 within a pair). For the 3-arm test, total = `3 x n_per_arm`, and each arm accrues `DAILY_ELIGIBLE_VISITORS / 3` per day, so `days = n_per_arm / (daily / 3)`.

In [4]:
from cProfile import label


def size_table(rel_grid, baseline, daily, label):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[0.5],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total_3arm'] = df['n_per_arm'] * N_ARMS
    if daily:
        df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
        df['weeks_required'] = (df['days_required'] / 7).round(1)
        cols = ['rel_effect','p_treatment','n_per_arm','n_total_3arm','days_required','weeks_required']
    else:
        cols = ['rel_effect','p_treatment','n_per_arm','n_total_3arm']
    
    sided = 'two-sided' if TWO_SIDED else 'one-sided'
    print(f"--- {label} (baseline={baseline:.1%}, alpha={ALPHA} Bonferroni, power={POWER:.0%}, {sided}) ---")

    return df[cols]

size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE_VISITORS, 'Positive MDE')

--- Positive MDE (baseline=52.8%, alpha=0.025 Bonferroni, power=80%, two-sided) ---


,rel_effect,p_treatment,n_per_arm,n_total_3arm,days_required,weeks_required
0,+3%,0.543376,18878,56634,7,1.0
1,+5%,0.553927,6784,20352,3,0.4
2,+8%,0.569753,2642,7926,1,0.1
3,+10%,0.580304,1687,5061,1,0.1


In [5]:
# Harm side (Early Stop Condition). With a two-sided test these n's mirror the positive grid;
# shown explicitly to document the harm magnitude the test is powered to detect.
size_table(HARM_GRID, BASELINE_RATE, DAILY_ELIGIBLE_VISITORS, 'Harm / guardrail')

--- Harm / guardrail (baseline=52.8%, alpha=0.025 Bonferroni, power=80%, two-sided) ---


,rel_effect,p_treatment,n_per_arm,n_total_3arm,days_required,weeks_required
0,-3%,0.511723,18944,56832,7,1.0
1,-5%,0.501172,6824,20472,3,0.4


## Step 3 — Summary for Experiment Design doc

The MDE should be defined in advance, then pasted to document the Power Analysis section in the Experiment Design doc.

In [6]:
TARGET_REL_MDE = 0.03 # NOTE: this is the relative MDE we are sizing for (3% relative lift on signup-page conversion)

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[0.5],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED
    )

n_per_arm = int(list(res.values())[0]['n_treatment'])

duration = f"{int(np.ceil(n_per_arm / (DAILY_ELIGIBLE_VISITORS/N_ARMS)))} days" if DAILY_ELIGIBLE_VISITORS else 'TODO (run Step 1)'

summary = {
    'Metric Used':                 'Visit-to-Signup Rate (VSR)',
    'Baseline Value':              f"{BASELINE_RATE:.1%} signup-page conversion",
    'Minimum Detectable Effect':   f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE:.3f} -> {BASELINE_RATE*(1+TARGET_REL_MDE):.3f})",
    'One/Two-Sided Test':          'Two-sided',
    'Significance Level':          f"{INITIAL_ALPHA} initial alpha; {ALPHA} per comparison (Bonferroni, m={N_COMPARISONS})",
    'Statistical Power':           f"{POWER:.0%}",
    'Variant Split %':             '33% / 33% / 33% (Control / A / B)',
    'Minimum Samples by Variant':  f"{n_per_arm:,}",
    'Minimum Samples total':       f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required':  duration,
    'Early Stop / harm threshold': f"-{abs(TARGET_REL_MDE):.0%} relative (covered by two-sided sizing)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,Visit-to-Signup Rate (VSR)
Baseline Value,52.8% signup-page conversion
Minimum Detectable Effect,+3% relative (0.528 -> 0.543)
One/Two-Sided Test,Two-sided
Significance Level,0.05 initial alpha; 0.025 per comparison (Bonf...
Statistical Power,80%
Variant Split %,33% / 33% / 33% (Control / A / B)
Minimum Samples by Variant,"18,878"
Minimum Samples total,"56,634"
Shortest Duration Required,7 days
